<a href="https://colab.research.google.com/github/T-ishiba/ProteinName2structure/blob/main/PreviousStructures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython requests

import requests
import os
from Bio.PDB import PDBList
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

def get_pdb_ids_from_uniprot(query: str) -> list:
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": query, "format": "json"}
    print(f"Searching UniProt for '{query}'...")

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        result = response.json()
    except requests.exceptions.RequestException as e:
        print(f"API Request Error: {e}")
        return []

    if not result.get('results'):
        print("No results found.")
        return []

    first_hit = result['results'][0]
    accession = first_hit.get("primaryAccession", "Unknown")
    print(f"Top hit Accession: {accession}")

    cross_refs = first_hit.get('uniProtKBCrossReferences', [])
    pdb_ids = [ref['id'] for ref in cross_refs if ref['database'] == "PDB"]
    print(f"Found {len(pdb_ids)} associated PDB IDs.\n")

    return pdb_ids

def download_pdb_files(pdb_ids: list, download_dir: str):
    if not pdb_ids:
        return
    os.makedirs(download_dir, exist_ok=True)
    pdbl = PDBList()
    print(f"=== Downloading mmCIF files to {download_dir} ===")
    pdbl.download_pdb_files(pdb_ids, pdir=download_dir, file_format="mmCif", overwrite=True)
    print("Download completed.\n")

def extract_mmcif_metadata(pdb_ids: list, download_dir: str, output_filename: str = "protein_data.tsv"):
    print("=== Extracting structural metadata and saving to TSV ===")

    output_filepath = os.path.join(download_dir, output_filename)

    with open(output_filepath, mode="w", encoding="utf-8") as f:
        # Write TSV header
        f.write("PDB_ID\tDate\tOrganism\tTitle\tMethod\tResolution\tProteins\tLigands\tDOI\n")

        for pdb_id in pdb_ids:
            filename = f"{pdb_id.lower()}.cif"
            filepath = os.path.join(download_dir, filename)

            if not os.path.exists(filepath):
                print(f"File not found: {filepath}")
                continue

            try:
                mmcif_dict = MMCIF2Dict(filepath)

                # Extract basic data
                title = mmcif_dict.get("_struct.title", ["Unknown"])[0]
                method = mmcif_dict.get("_exptl.method", ["Unknown"])[0]
                resolution = mmcif_dict.get("_reflns.d_resolution_high", ["N/A"])[0]
                doi = mmcif_dict.get("_citation.pdbx_database_id_DOI", ["N/A"])[0]

                # Extract Deposition Date
                dep_date = mmcif_dict.get("_pdbx_database_status.recvd_initial_deposition_date", ["Unknown"])[0]

                # Extract Organism (combining natural and genetically engineered sources)
                org_nat = mmcif_dict.get("_entity_src_nat.pdbx_organism_scientific", [])
                if isinstance(org_nat, str): org_nat = [org_nat]

                org_gen = mmcif_dict.get("_entity_src_gen.pdbx_gene_src_scientific_name", [])
                if isinstance(org_gen, str): org_gen = [org_gen]

                clean_orgs = [o for o in (org_nat + org_gen) if o and o not in ["?", "Unknown"]]
                organism = ", ".join(set(clean_orgs)) if clean_orgs else "Unknown"

                # Extract Proteins (excluding water molecules)
                proteins_raw = mmcif_dict.get("_entity.pdbx_description", [])
                if isinstance(proteins_raw, str): proteins_raw = [proteins_raw]
                clean_proteins = [p.replace('\n', ' ').replace('\r', '') for p in proteins_raw if p.lower() not in ["water", "hoh", "h2o"]]
                proteins = ", ".join(set(clean_proteins)) if clean_proteins else "Unknown"

                # Extract Ligands (excluding water molecules)
                ligands_raw = mmcif_dict.get("_pdbx_entity_nonpoly.name", [])
                if isinstance(ligands_raw, str): ligands_raw = [ligands_raw]
                clean_ligands = [l.replace('\n', ' ').replace('\r', '') for l in ligands_raw if l.lower() not in ["water", "hoh", "h2o"]]
                ligands = ", ".join(clean_ligands) if clean_ligands else "None"

                # Console output for verification
                print(f"[{pdb_id.upper()}]")
                print(f"  Date      : {dep_date}")
                print(f"  Organism  : {organism}")
                print(f"  Title     : {title}")
                print(f"  Method    : {method}")
                print(f"  Resolution: {resolution} Å")
                print(f"  Proteins  : {proteins}")
                print(f"  Ligands   : {ligands}")
                print("-" * 40)

                # Write to TSV
                clean_title = title.replace('\n', ' ').replace('\r', '')
                f.write(f"{pdb_id.upper()}\t{dep_date}\t{organism}\t{clean_title}\t{method}\t{resolution}\t{proteins}\t{ligands}\t{doi}\n")

            except Exception as e:
                print(f"Error parsing {pdb_id}: {e}")

    print(f"\n★ Metadata successfully saved to '{output_filepath}'.")

# ==========================================
# Main Execution Block
# ==========================================
if __name__ == "__main__":
    # Prompt user for input
    target_name = input("Enter target protein and organism (e.g., FOXP3&&human): ").strip()

    # Create a safe folder name from the query
    safe_folder_name = target_name.replace("&&", "_").replace(" ", "_")
    download_directory = f"./pdb_files/{safe_folder_name}"

    pdb_list = get_pdb_ids_from_uniprot(target_name)

    if pdb_list:
        download_pdb_files(pdb_list, download_dir=download_directory)
        extract_mmcif_metadata(pdb_list, download_dir=download_directory, output_filename="protein_data.tsv")

In [ ]:
import os
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio import Align
import matplotlib.pyplot as plt

def extract_sequences_from_cif(filepath: str) -> list:
    """Extract amino acid sequences from a cif file."""
    try:
        mmcif_dict = MMCIF2Dict(filepath)
        seqs = mmcif_dict.get("_entity_poly.pdbx_seq_one_letter_code", [])
        chain_ids = mmcif_dict.get("_entity_poly.pdbx_strand_id", [])

        extracted_data = []
        for chain_id, seq in zip(chain_ids, seqs):
            clean_seq = seq.replace('\n', '').replace(' ', '').replace('\r', '')
            extracted_data.append({"chain": chain_id, "sequence": clean_seq})

        return extracted_data
    except Exception as e:
        print(f"File reading error ({filepath}): {e}")
        return []

# ==========================================
# Main Execution Block
# ==========================================
if __name__ == "__main__":
    # 1. Input the reference full-length amino acid sequence
    query_sequence = input("Enter the reference full-length amino acid sequence: ").strip().upper()
    query_length = len(query_sequence)

    # 2. Specify the target directory containing cif files
    input_dir = input("Enter the path to the directory containing cif files (default: ./pdb_files): ").strip()
    cif_directory = input_dir if input_dir else "./pdb_files"

    output_image_path = os.path.join(cif_directory, "sequence_coverage_map_with_seq.png")

    if not os.path.exists(cif_directory):
        print(f"Error: Directory '{cif_directory}' not found. Please check the path.")
    else:
        print(f"\nDrawing sequence coverage map (this may take a moment depending on sequence length)...\n")

        # Initialize the aligner (Local alignment)
        aligner = Align.PairwiseAligner()
        aligner.mode = 'local'
        aligner.match_score = 2
        aligner.mismatch_score = -1
        aligner.open_gap_score = -5
        aligner.extend_gap_score = -1

        results = []

        # Process all cif files in the specified directory
        for filename in sorted(os.listdir(cif_directory)):
            if filename.endswith(".cif"):
                filepath = os.path.join(cif_directory, filename)
                pdb_id = filename.split(".")[0].upper()

                target_data_list = extract_sequences_from_cif(filepath)

                for data in target_data_list:
                    chain_name = data['chain']
                    target_seq = data['sequence']
                    display_name = f"{pdb_id}_{chain_name}"

                    # Perform local alignment
                    alignments = aligner.align(query_sequence, target_seq)
                    if len(alignments) > 0:
                        best_alignment = alignments[0]
                        matched_blocks_on_query = best_alignment.aligned[0]

                        results.append({
                            "name": display_name,
                            "blocks": matched_blocks_on_query
                        })

        if not results:
            print("No comparison targets found.")
        else:
            # --- Draw the plot ---
            num_targets = len(results)

            # Dynamically set width to allocate approx 0.15 inches per amino acid
            fig_width = max(12, query_length * 0.15)
            # Calculate height including the query sequence row
            fig_height = max(3.5, (num_targets + 1) * 0.6)

            fig, ax = plt.subplots(figsize=(fig_width, fig_height))

            y_labels = []
            y_ticks = []

            # --- Draw Query Sequence on the top row ---
            query_y_pos = num_targets + 1
            y_labels.append("Query\nSequence")
            y_ticks.append(query_y_pos)

            # Background guideline for the full length
            ax.hlines(y=query_y_pos, xmin=0, xmax=query_length, linewidth=18, color='lightgray', alpha=0.4)

            # Plot each amino acid at its exact coordinate
            for i, aa in enumerate(query_sequence):
                ax.text(i + 0.5, query_y_pos, aa, ha='center', va='center', fontsize=10, fontfamily='monospace', fontweight='bold')

            # --- Draw coverage for each cif file ---
            for i, res in enumerate(results):
                y_pos = num_targets - i
                y_labels.append(res["name"])
                y_ticks.append(y_pos)

                # Faint guideline for readability
                ax.hlines(y=y_pos, xmin=0, xmax=query_length, linewidth=1, color='lightgray', alpha=0.5, linestyle=':')

                # Draw thick blue lines for matched blocks
                for (start, end) in res["blocks"]:
                    ax.hlines(y=y_pos, xmin=start, xmax=end, linewidth=8, color='royalblue')

            # Format the plot aesthetics
            ax.set_yticks(y_ticks)
            ax.set_yticklabels(y_labels, fontsize=12, fontweight='bold')
            ax.set_xlim(0, query_length)
            ax.set_ylim(0.5, num_targets + 1.8)

            # Set x-axis ticks every 50 residues
            ax.set_xticks(range(0, query_length + 1, 50))
            ax.tick_params(axis='x', labelsize=12)

            ax.set_xlabel("Amino Acid Position", fontsize=14)
            ax.set_title("Sequence Coverage Map with Amino Acid Residues", fontsize=18, pad=20)

            # Remove plot borders
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)

            # Save the image with tight bounding box
            plt.savefig(output_image_path, dpi=150, bbox_inches='tight')
            plt.close(fig)

            print(f"★ Sequence coverage map saved to: {output_image_path}")